# Overview

Notebook for building out nanoGPT


In [1]:
import math

import torch
import torch.nn.functional as F
from torch import nn

/Users/davidwon/projects/nanogpt/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


# Data Loading and Processing


Download data


In [2]:
with open("input.txt") as file:
    words = file.read()

In [3]:
len(words)

1115394

Extract vocab (character-level tokenization)


In [4]:
vocab = sorted(set(words))
print(vocab)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [5]:
vocab_size = len(vocab)
vocab_size

65

Maps for going from character to index and vice versa


In [6]:
# Would rather name t_to_i or c_to_i but doing this to be consistent with video
s_to_i = {s: i for i, s in enumerate(vocab)}
i_to_s = {i: s for i, s in enumerate(vocab)}

encode = lambda seq: [s_to_i[ch] for ch in seq]
decode = lambda idxs: "".join([i_to_s[idx] for idx in idxs])

encoded = encode(words[:10])
print(f"Original: {words[:10]}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decode(encoded)}")

Original: First Citi
Encoded: [18, 47, 56, 57, 58, 1, 15, 47, 58, 47]
Decoded: First Citi


Convert data to tensor


In [7]:
data = torch.tensor(encode(words))
data.shape

torch.Size([1115394])

In [8]:
data[:10].tolist() == encode(words[:10])

True

Train/val split


In [9]:
partition_idx = int(len(data) * 0.9)
train_data, val_data = data[:partition_idx], data[partition_idx:]
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

Code for getting and visualizing batches


In [10]:
# Seed 1337 to match video
torch.manual_seed(1337)
batch_size = 4
block_size = 8


def get_batch(split):
    if split == "train":
        idx = torch.randint(len(train_data) - block_size, (batch_size,))
        data = train_data
    elif split == "val":
        idx = torch.randint(len(val_data) - block_size, (batch_size,))
        data = val_data
    else:
        raise ValueError("``split`` must be 'train' or 'val'")

    X, y = [], []
    for i in idx:
        X.append(data[i : i + block_size])
        y.append(data[i + 1 : i + 1 + block_size])
    X = torch.stack(X)
    y = torch.stack(y)

    return X, y


Xb, yb = get_batch("train")
print("Xb:")
print(Xb.shape)
print(Xb)
print("yb:")
print(yb.shape)
print(yb)

Xb:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
yb:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [11]:
x_sample, y_sample = Xb[0], yb[0]
input = []
for j in range(block_size):
    input.append(x_sample[j])
    print(f"When the input is {input}, the target is {y_sample[j]}")

When the input is [tensor(24)], the target is 43
When the input is [tensor(24), tensor(43)], the target is 58
When the input is [tensor(24), tensor(43), tensor(58)], the target is 5
When the input is [tensor(24), tensor(43), tensor(58), tensor(5)], the target is 57
When the input is [tensor(24), tensor(43), tensor(58), tensor(5), tensor(57)], the target is 1
When the input is [tensor(24), tensor(43), tensor(58), tensor(5), tensor(57), tensor(1)], the target is 46
When the input is [tensor(24), tensor(43), tensor(58), tensor(5), tensor(57), tensor(1), tensor(46)], the target is 43
When the input is [tensor(24), tensor(43), tensor(58), tensor(5), tensor(57), tensor(1), tensor(46), tensor(43)], the target is 39


In [12]:
# fmt: off
x_sample, y_sample = Xb[0], yb[0]
input = []
for j in range(block_size):
    input.append(x_sample[j].item())
    print(
        f"When the input is {decode(input)}, the target is {decode([y_sample[j].item()])}"
    )

When the input is L, the target is e
When the input is Le, the target is t
When the input is Let, the target is '
When the input is Let', the target is s
When the input is Let's, the target is  
When the input is Let's , the target is h
When the input is Let's h, the target is e
When the input is Let's he, the target is a


# Build Model


Simple bigram model


In [13]:
yb.shape

torch.Size([4, 8])

In [ ]:
# fmt: off
torch.manual_seed(1337)

class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Straight idx to logits
        self.embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        if targets is None:
            return self.embedding(idx), None

        logits = self.embedding(idx) # (B, L, C) or (batch_size, block_size, vocab_size) in this case

        B, L, C = logits.shape
        # Reshaping to match expected shape of ``F.cross_entropy()``
        loss = F.cross_entropy(logits.reshape((B * L, C)), targets.reshape(B * L))
        return logits, loss

    @torch.no_grad()
    def generate(self, input, n_new_tokens):
        """
        input is a tensor of shape (batch_size, seq_length)
        """
        for _ in range(n_new_tokens):
            logits = self(input) # (B, L, C)
            # Only need the predictions for the next char (idx)
            probs = F.softmax(logits[:, -1, :], dim=-1) # (B, C)
            next_idx = torch.multinomial(probs, 1)
            input = torch.cat([input, next_idx], dim=1)
        
        return input


model = BigramLM(vocab_size)
logits, loss = model(Xb, yb)
print(f"Loss: {loss}")
print(f"Loss with uniform sampling: {math.log(vocab_size)}")

# Generate new sequence
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long), 100)[0].tolist()))

Loss: 4.878634929656982
Loss with uniform sampling: 4.174387269895637

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [15]:
input = [1]
logits = model(torch.tensor([input]))
logits.shape

torch.Size([1, 1, 65])

# Train Model


In [42]:
n_steps = 1000

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(n_steps):
    Xb, yb = get_batch("train")
    optimizer.zero_grad()
    _, loss = model(Xb, yb)
    loss.backward()
    optimizer.step()

# Confirm that model is learning
print(loss.item())

2.4656097888946533
